# 🧠 Taller 3: Clínica Psiquiátrica IA — Técnicas de Prompt Engineering

Hoy trabajamos en una **clínica psiquiátrica** y nuestra herramienta de apoyo es un **modelo de lenguaje**. Dependiendo de cómo le pidamos las cosas, nos dará respuestas genéricas o análisis profesionales.

Cada técnica de prompting es una herramienta clínica diferente:

| Técnica | Herramienta clínica | Ejemplo |
|---|---|---|
| **Zero Shot** | Redacción clínica | Mejorar un diagnóstico mal escrito |
| **Few Shot** | Historial clínico | Casos similares como referencia |
| **Chain of Thought** | Evaluación diagnóstica | Análisis paso a paso |
| **Role Prompting** | Junta médica | Diferentes especialistas opinan |

⏱️ **Tiempo estimado:** 30 minutos

> ⚠️ **Nota:** Este taller es con fines educativos sobre prompt engineering. La IA no reemplaza profesionales de salud mental.

In [2]:
pip install ollama openai

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## ⚙️ Opcional: Ejecución en Local con Ollama

> Si cuentas con el hardware necesario, puedes ejecutar este taller de forma **local y sin conexión a internet** usando [Ollama](https://ollama.com),  tal cual lo vimos en la clase anterior.

Para ello, ejecuta los siguientes bloques alternativos **en lugar de** los bloques con la librería de OpenAI.

**Requisitos:**
- Tener [Ollama](https://ollama.com) instalado
- Haber descargado el modelo: `ollama pull gemma2:2b`

In [1]:
import ollama
from IPython.display import display, Markdown
def consultar_ollama(prompt, opciones=None, modelo="gemma2:2b"):
    respuesta = ollama.generate(
        model=modelo,
        prompt=prompt,
        options=opciones or {},
        think=False
    )
    print("=" * 60)
    print(f"Parametros: {opciones or 'default'}")
    print("=" * 60)
    print(respuesta["response"])
    print(f"\n[Tokens generados: {respuesta.get('eval_count', 'N/A')}]")
    return respuesta["response"]

print("Consultorio listo. Sistema IA conectado.")

Consultorio listo. Sistema IA conectado.


In [3]:
# Prueba de conexion
respuesta = ollama.generate(
    model="gemma2:2b",
    prompt="Necesito que respondas unicamente el siguiente texto 'Consultorio Psicologico IA listo' y nada mas.",
    think=False
)
print(respuesta["response"])

Consultorio Psicologico IA listo 



## ☁️ Ejecución en la Nube con Google Gemini

> Si no tienes hardware disponible, puedes ejecutar este taller de forma gratuita desde cualquier navegador usando la **API de Google Gemini**, compatible con la librería de OpenAI.

### 🔑 ¿Cómo obtener tu API Key?

1. Ve a [Google AI Studio](https://aistudio.google.com/apikey)
2. Inicia sesión con tu cuenta de Google
3. Haz clic en **"Create API Key"**
4. Copia la clave generada y pégala en la celda de configuración de abajo

> 💡 La clave es gratuita y no requiere tarjeta de crédito para el nivel básico de uso.

In [22]:
from openai import OpenAI
from IPython.display import display, Markdown
client = OpenAI(
    api_key="TU_GEMINI_API_KEY",  # 👈 Reemplaza aquí
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

def consultar(prompt, opciones=None, modelo="gemini-3.1-flash-lite"):
    opciones = opciones or {}
    respuesta = client.chat.completions.create(
        model=modelo,
        messages=[{"role": "user", "content": prompt}],
        temperature=opciones.get("temperature", 1.0),
        max_tokens=opciones.get("num_predict", 2048)
    )
    texto = respuesta.choices[0].message.content
    tokens = respuesta.usage.completion_tokens if respuesta.usage else "N/A"
    print("=" * 60)
    print(f"Parametros: {opciones or 'default'}")
    print("=" * 60)
    display(Markdown(texto))
    print(f"\n[Tokens generados: {tokens}]")
    

print("Consultorio listo. Modo NUBE con Google Gemini activado.")

Consultorio listo. Modo NUBE con Google Gemini activado.


## 1️⃣ Zero Shot: Redacción Clínica

Vamos pedirle al modelo que haga algo **sin darle ejemplos**. Solo con la instrucción.

📝 Un residente escribió este diagnóstico a las 3am después de una guardia. Vamos a pedirle a la IA que lo mejore.

In [23]:
# ZERO SHOT: Mejorar la redaccion de un diagnostico
diagnostico_mal_escrito = """paciente juan lopez, 32 años vino pq no puede dormir
hace como 3 semanas y dice q esta re ansioso en el camello, no tiene historial
de nada psiquiatrico. se le ve nervioso y habla rapido. creo q es ansiedad
generalizada o algo asi, le mande unas pastillas para dormir y que vuelva
en 2 semanas a ver como sigue."""

prompt_zero_shot = f"""Eres un psiquiatra clinico redactando informes para el expediente
de la clinica.

El siguiente diagnostico fue escrito de forma informal por un pasante de psiquiatria.
Reescribelo como un informe clinico profesional.

Diagnostico original:
{diagnostico_mal_escrito}
"""

print("ZERO SHOT - Mejorar redaccion de diagnostico")
consultar(prompt_zero_shot, opciones={"temperature": 0.3, "num_predict": 2000})

ZERO SHOT - Mejorar redaccion de diagnostico
Parametros: {'temperature': 0.3, 'num_predict': 2000}


Aquí tienes una propuesta de redacción formal, estructurada bajo los estándares de una nota de evolución clínica:

***

**INFORME CLÍNICO DE EVALUACIÓN PSIQUIÁTRICA**

**Paciente:** Juan López
**Edad:** 32 años
**Fecha de consulta:** [Insertar fecha]

**Motivo de consulta:** Insomnio y sintomatología ansiosa.

**Anamnesis:**
Paciente masculino de 32 años de edad, sin antecedentes psiquiátricos de relevancia, quien acude a consulta refiriendo cuadro clínico de aproximadamente tres semanas de evolución caracterizado por insomnio (dificultad para la conciliación del sueño) y sintomatología ansiosa exacerbada en el entorno laboral. Niega antecedentes médicos de importancia o consumo de sustancias.

**Examen del estado mental:**
A la exploración, el paciente se muestra consciente, orientado y colaborador. Se observa inquietud psicomotriz, con un discurso acelerado (taquilalia) y signos evidentes de ansiedad manifiesta. No se observan alteraciones en el contenido del pensamiento ni ideación delirante o alucinatoria en el momento de la entrevista.

**Impresión diagnóstica:**
*   **F41.1 Trastorno de Ansiedad Generalizada (CIE-10/DSM-5):** A confirmar tras seguimiento.
*   **G47.0 Insomnio de conciliación.**

**Plan de tratamiento:**
1.  **Farmacoterapia:** Se prescribe tratamiento hipnótico/ansiolítico a dosis bajas para el manejo sintomático del insomnio [especificar fármaco y dosis, ej: Clonazepam 0.5mg o Zolpidem 5mg, según criterio].
2.  **Seguimiento:** Se cita al paciente en dos semanas para reevaluar la respuesta al tratamiento, monitorear la evolución de los síntomas ansiosos y descartar otros diagnósticos diferenciales.
3.  **Recomendaciones:** Se brindan pautas de higiene del sueño.

***

**Firma:**
[Tu Nombre/Firma]
Psiquiatría Clínica


[Tokens generados: 451]


## 2️⃣ Few Shot: Historial Clínico

En muchos casos, varias industrias estandarizan formatos para facilitar la lectura y auditoria de los casos, ahora daremos **ejemplos de casos anteriores** antes de pedirle que analice un caso nuevo para que La IA aprende el patrón y nos de en el formato que usa clínica.

👤 **Paciente Pedro:** 45 años, se siente triste y sin motivación desde hace 2 meses. Vamos a darle casos similares para que formule una recomendación consistente.

In [24]:
# FEW SHOT: Paciente Pedro con historial de casos similares
prompt_pedro = """Eres un psicologo clinico. Analiza pacientes y formula recomendaciones
siguiendo el formato de estos casos anteriores:

Caso 1 - Maria, 38 anos: Fatiga constante, perdida de interes en actividades.
Diagnostico: Episodio depresivo leve.
Recomendacion: Terapia cognitivo-conductual (12 sesiones) + rutina de ejercicio 3x/semana.

Caso 2 - Carlos, 52 anos: Irritabilidad, dificultad para concentrarse, dolores de cabeza.
Diagnostico: Sindrome de burnout.
Recomendacion: Licencia laboral temporal + tecnicas de manejo de estres + evaluacion en 4 semanas.

Caso 3 - Lucia, 29 anos: Llanto frecuente, aislamiento social, perdida de apetito tras ruptura sentimental.
Diagnostico: Trastorno adaptativo con animo depresivo.
Recomendacion: Terapia de apoyo semanal + actividades de reconexion social + seguimiento en 6 semanas.

Ahora analiza este nuevo paciente:

Paciente Pedro, 45 anos: Se siente triste y sin motivacion desde hace 2 meses.
Ha dejado de hacer ejercicio, duerme mas de lo normal y ha perdido interes en
su trabajo como profesor. No identifica un evento desencadenante claro.

Usa el mismo formato: Diagnostico + Recomendacion."""

print("FEW SHOT - Paciente Pedro (tristeza y desmotivacion)")
print()
consultar(prompt_pedro, opciones={"temperature": 0.3, "num_predict": 200})

FEW SHOT - Paciente Pedro (tristeza y desmotivacion)

Parametros: {'temperature': 0.3, 'num_predict': 200}


Caso 4 - Pedro, 45 años: Tristeza persistente, anhedonia, hipersomnia y falta de motivación laboral sin causa aparente.

Diagnóstico: Trastorno depresivo mayor (episodio único, moderado).

Recomendación: Psicoterapia cognitivo-conductual (foco en activación conductual) + interconsulta con psiquiatría para valoración farmacológica + higiene del sueño estricta.


[Tokens generados: 93]


## 3️⃣ Chain of Thought: Evaluación Diagnóstica

🔗 **Chain of Thought** = pedirle al modelo que **razone paso a paso** antes de dar su conclusión.

👵 **Paciente Rosa:** 68 años, su familia reporta que olvida cosas frecuentemente. ¿Es demencia o envejecimiento normal? La IA debe evaluar con 3 preguntas clínicas y razonar antes de concluir.

In [25]:
# CHAIN OF THOUGHT: Evaluacion de paciente Rosa
prompt_rosa = """Eres un neuropsicologo clinico evaluando un posible caso de deterioro cognitivo.

Paciente: Rosa, 68 anos. Su familia reporta que olvida nombres de familiares cercanos,
pierde objetos cotidianos (llaves, telefono) y la semana pasada se desoriento
volviendo del supermercado, un camino que hace desde hace 20 anos.

Piensa paso a paso:

Paso 1: Formula 3 preguntas clinicas clave que le harias a Rosa y su familia
para diferenciar envejecimiento normal de deterioro cognitivo.

Paso 2: Para cada pregunta, explica que respuesta indicaria envejecimiento normal
y cual indicaria deterioro cognitivo.

Paso 3: Basandote en los sintomas descritos (olvido de nombres cercanos,
perdida de objetos, desorientacion en ruta familiar), da tu evaluacion preliminar.

Paso 4: Recomienda los siguientes pasos (pruebas, derivaciones, seguimiento).

Formato: Enumera cada paso claramente y explica detalladamente su justificacion.
Restriccion: Se cauteloso, esto es una evaluacion preliminar, no un diagnostico definitivo."""

print("CHAIN OF THOUGHT - Paciente Rosa (sospecha de deterioro cognitivo)")
print()
consultar(prompt_rosa, opciones={"temperature": 0.4, "num_predict": 2000})

CHAIN OF THOUGHT - Paciente Rosa (sospecha de deterioro cognitivo)

Parametros: {'temperature': 0.4, 'num_predict': 2000}


Como neuropsicólogo clínico, abordo este caso con la cautela necesaria, entendiendo que los síntomas reportados son signos de alerta que requieren una investigación exhaustiva para determinar si estamos ante un proceso neurodegenerativo, una condición reversible o un envejecimiento con declive leve.

Aquí presento el análisis preliminar:

---

### Paso 1: Preguntas clínicas clave

Para diferenciar entre el olvido benigno asociado a la edad y un posible deterioro cognitivo, realizaría las siguientes preguntas:

1.  **A la familia:** ¿Estos olvidos y episodios de desorientación han interferido con su capacidad para realizar actividades de la vida diaria (gestionar dinero, cocinar, seguir una receta, usar electrodomésticos)?
2.  **A Rosa:** Cuando olvida un nombre o pierde un objeto, ¿es capaz de recordarlo más tarde por sí misma o mediante pistas (memoria de reconocimiento), o el dato desaparece por completo de su registro?
3.  **A la familia:** ¿Ha habido cambios notables en su estado de ánimo, iniciativa, personalidad o comportamiento social en los últimos 6 a 12 meses?

---

### Paso 2: Diferenciación de respuestas

*   **Pregunta 1 (Funcionalidad):**
    *   *Envejecimiento normal:* La persona puede tener olvidos, pero mantiene su autonomía. Si se pierde, logra reorientarse o pedir ayuda de forma lógica.
    *   *Deterioro cognitivo:* Existe una pérdida de funcionalidad (incapacidad para completar tareas complejas que antes dominaba). La desorientación en rutas conocidas es un marcador de alarma de afectación cortical.
*   **Pregunta 2 (Recuperación de información):**
    *   *Envejecimiento normal:* El olvido suele ser un problema de "recuperación" (el nombre está en la punta de la lengua y aparece después).
    *   *Deterioro cognitivo:* El olvido es un problema de "almacenamiento" (la información no se consolidó). Si no recuerda el nombre ni siquiera con pistas, sugiere un fallo en la memoria episódica.
*   **Pregunta 3 (Síntomas neuropsiquiátricos):**
    *   *Envejecimiento normal:* La personalidad y el juicio permanecen estables.
    *   *Deterioro cognitivo:* Cambios como apatía, irritabilidad o pérdida de iniciativa suelen preceder o acompañar a los déficits cognitivos en demencias degenerativas.

---

### Paso 3: Evaluación preliminar

Basado en la descripción, los síntomas de Rosa son **altamente sugestivos de un deterioro cognitivo patológico**, posiblemente de tipo amnésico.

*   **Justificación:** El olvido de nombres de familiares cercanos (memoria semántica/episódica) y, especialmente, la **desorientación espacial en una ruta habitual** (memoria procedimental y visuoespacial), son indicadores de que el proceso ha superado los límites de los "olvidos benignos del envejecimiento". La desorientación en un entorno conocido sugiere una afectación en estructuras temporales mediales (como el hipocampo) y áreas de asociación parietal, lo cual requiere una evaluación urgente para descartar un proceso neurodegenerativo (como una Enfermedad de Alzheimer en fase prodrómica) o causas secundarias.

---

### Paso 4: Recomendaciones y pasos a seguir

Como profesional, mi recomendación es actuar con celeridad pero sin alarmismo, siguiendo este protocolo:

1.  **Evaluación Neuropsicológica Completa:** Aplicar pruebas estandarizadas (como el *MoCA* o *MMSE* para cribado, seguidos de una batería neuropsicológica profunda: *Test de Aprendizaje Verbal de Rey*, *Figura Compleja de Rey*, *Test de Boston*, etc.) para mapear qué dominios cognitivos están afectados.
2.  **Valoración Médica (Neurología/Geriatría):**
    *   **Analítica completa:** Descartar causas reversibles (déficit de vitamina B12, hipotiroidismo, infecciones, efectos secundarios de fármacos o depresión pseudodemencial).
    *   **Neuroimagen:** Solicitar una Resonancia Magnética (RM) cerebral para evaluar atrofia hipocampal o lesiones vasculares.
3.  **Evaluación Funcional:** Aplicar escalas de Actividades de la Vida Diaria (Lawton y Brody) para cuantificar el grado de dependencia actual.
4.  **Seguimiento:** Si los resultados son equívocos, se recomienda un seguimiento a los 3-6 meses para observar la tasa de progresión de los síntomas.

**Nota importante:** *Esta evaluación es preliminar. Los síntomas descritos requieren una confirmación clínica mediante pruebas objetivas antes de realizar cualquier diagnóstico. Es fundamental que la familia acompañe a Rosa en este proceso para reducir su ansiedad y asegurar la precisión de la información clínica.*


[Tokens generados: 1061]


## 4️⃣ Role Prompting: Junta Médica

🎭 **Role Prompting** = asignarle un **rol específico** al modelo. La misma situación vista por diferentes especialistas produce análisis muy distintos.

👩‍💻 **Paciente Ana:** 25 años, miedo intenso a hablar en público que le está afectando su carrera. Veamos qué dice cada especialista.

In [26]:
# ROLE PROMPTING: 3 especialistas evaluan a la paciente Ana
caso_ana = """Paciente: Ana, 25 anos, ingeniera de software. Tiene miedo intenso a hablar
en publico. Evita presentaciones en el trabajo, rechaza ascensos que impliquen liderar
reuniones, y la semana pasada tuvo un ataque de panico antes de una exposicion."""

# Especialista 1: Psicologo cognitivo-conductual
prompt_psicologo = f"""Eres un psicologo especialista en terapia cognitivo-conductual (TCC)
con enfoque en fobias y ansiedad.

{caso_ana}

Da tu analisis y plan de tratamiento en maximo 4 oraciones."""

print("ESPECIALISTA 1: Psicologo Cognitivo-Conductual")
print()
consultar(prompt_psicologo, opciones={"temperature": 0.5, "num_predict": 500})

print("\n" + "~" * 60 + "\n")

# Especialista 2: Psiquiatra
prompt_psiquiatra = f"""Eres un psiquiatra con enfoque farmacologico y neurobiologico.

{caso_ana}

Da tu analisis y plan de tratamiento en maximo 4 oraciones."""

print("ESPECIALISTA 2: Psiquiatra")
print()
consultar(prompt_psiquiatra, opciones={"temperature": 0.5, "num_predict": 500})

print("\n" + "~" * 60 + "\n")

# Especialista 3: Coach de vida
prompt_coach = f"""Eres un coach de vida y oratoria con experiencia ayudando a profesionales
a superar el miedo escenico. Tu enfoque es practico y motivacional.

{caso_ana}

Da tu analisis y plan de accion en maximo 4 oraciones."""

print("ESPECIALISTA 3: Coach de Vida y Oratoria")
print()
consultar(prompt_coach, opciones={"temperature": 0.5, "num_predict": 500})

ESPECIALISTA 1: Psicologo Cognitivo-Conductual

Parametros: {'temperature': 0.5, 'num_predict': 500}


Hola Ana, tu fobia social se mantiene mediante un ciclo de evitación que refuerza la creencia de que el peligro es inminente, por lo que trabajaremos en reestructurar tus pensamientos catastróficos sobre el juicio ajeno. Implementaremos un plan de exposición gradual, comenzando por situaciones de baja ansiedad para habituar a tu sistema nervioso y reducir la respuesta de pánico. Paralelamente, utilizaremos técnicas de entrenamiento en respiración y desactivación fisiológica para que recuperes el control durante las presentaciones. El objetivo es que logres enfrentar estas situaciones de forma funcional, desvinculando el desempeño profesional de tu valor personal.


[Tokens generados: 133]

~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

ESPECIALISTA 2: Psiquiatra

Parametros: {'temperature': 0.5, 'num_predict': 500}


El cuadro de Ana sugiere un Trastorno de Ansiedad Social con síntomas de pánico, mediado por una hiperactividad de la amígdala y una desregulación en los circuitos serotoninérgicos y noradrenérgicos. Propongo iniciar un tratamiento con un ISRS (como sertralina) para la modulación a largo plazo de la respuesta al miedo, complementado con el uso puntual de un betabloqueante (propranolol) antes de exposiciones para bloquear los síntomas autonómicos. Este enfoque farmacológico debe integrarse obligatoriamente con terapia cognitivo-conductual enfocada en la exposición gradual para reestructurar las vías neuronales vinculadas a la evitación. El objetivo es reducir la reactividad del sistema nervioso simpático y facilitar la habituación ante el estímulo estresor.


[Tokens generados: 170]

~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

ESPECIALISTA 3: Coach de Vida y Oratoria

Parametros: {'temperature': 0.5, 'num_predict': 500}


Hola Ana, tu miedo no es una falta de capacidad, sino una respuesta fisiológica que podemos reprogramar mediante la exposición gradual y técnicas de regulación del sistema nervioso. Tu plan comenzará con ejercicios de respiración diafragmática antes de cada interacción y la práctica de presentaciones breves en entornos de confianza (como reuniones 1:1) para ganar seguridad técnica. Paralelamente, trabajaremos en reencuadrar tus pensamientos catastróficos, transformando la presión de "ser juzgada" en el propósito de "aportar valor" con tu conocimiento técnico. Recuerda que la confianza no es la ausencia de miedo, sino la decisión de actuar a pesar de él; vamos a dar el primer paso hoy.


[Tokens generados: 146]


## 5️⃣ Combinando Todo: Caso Complejo

🧩 Ahora combinamos **todas las técnicas** en un solo prompt: Role + Chain of Thought + formato estructurado.

In [27]:
# COMBINACION: Todas las tecnicas en un solo prompt
prompt_final = """Eres un psicologo clinico senior con 20 anos de experiencia y
especializacion en trastornos del animo. Trabajas en un hospital universitario.

CASO CLINICO:
Paciente Diego, 40 anos, empresario. Desde hace 3 meses presenta:
- Insomnio severo (duerme 2-3 horas por noche)
- Irritabilidad extrema con su familia y empleados
- Perdida de 8 kg sin dieta
- Episodios de llanto sin motivo aparente
- Consumo de alcohol incrementado (de social a diario)
- Verbaliza: "No le veo sentido a nada"

Casos de referencia en tu expediente:
- Paciente similar A: Sintomas parecidos tras divorcio -> Depresion mayor, respondio a TCC + ISRS.
- Paciente similar B: Sintomas parecidos + consumo de sustancias -> Trastorno depresivo con
  comorbilidad de abuso de sustancias, requirio internacion breve.

Piensa paso a paso:
Paso 1: Identifica los sintomas clave y su gravedad
Paso 2: Compara con los casos de referencia
Paso 3: Formula tu evaluacion preliminar
Paso 4: Disena un plan de tratamiento con prioridades

Formato: Secciones numeradas por cada paso.
Restriccion: Prioriza la seguridad del paciente. Esto es una evaluacion preliminar."""

print("CASO COMPLEJO: Paciente Diego (combinacion de tecnicas)")
print()
consultar(prompt_final, opciones={
    "temperature": 0.4,
    "num_predict": 2000,
    "repeat_penalty": 1.3
})

CASO COMPLEJO: Paciente Diego (combinacion de tecnicas)

Parametros: {'temperature': 0.4, 'num_predict': 2000, 'repeat_penalty': 1.3}


Como psicólogo clínico con dos décadas de experiencia en el ámbito hospitalario, presento la evaluación preliminar del caso de Diego. Dada la naturaleza de los síntomas y el riesgo asociado al consumo de sustancias y la ideación nihilista, este caso debe ser abordado con máxima prioridad clínica.

---

### Paso 1: Identificación de síntomas clave y gravedad
Los síntomas de Diego configuran un cuadro clínico de alta complejidad y severidad:
*   **Sintomatología vegetativa:** Insomnio severo (privación de sueño crónica que altera el juicio y la regulación emocional) y pérdida de peso significativa (8 kg en 3 meses), lo que sugiere un compromiso metabólico y una posible inhibición o agitación psicomotriz.
*   **Sintomatología afectiva:** Irritabilidad extrema (frecuentemente una presentación atípica de la depresión en varones) y labilidad emocional (llanto sin causa aparente).
*   **Sintomatología cognitiva:** Anhedonia y desesperanza ("No le veo sentido a nada"), lo que constituye una "bandera roja" para ideación suicida pasiva o activa.
*   **Conducta de riesgo:** Incremento en el consumo de alcohol (mecanismo de automedicación desadaptativo).
*   **Gravedad:** **Alta.** La combinación de insomnio, pérdida de peso, desesperanza y abuso de sustancias coloca al paciente en una zona de riesgo vital que requiere intervención inmediata.

### Paso 2: Comparación con casos de referencia
*   **Similitud con Caso A:** Diego comparte la sintomatología depresiva clásica. Sin embargo, en Diego observamos una mayor descompensación conductual (abuso de alcohol).
*   **Similitud con Caso B:** El perfil de Diego es casi idéntico al Caso B. La comorbilidad con el alcohol no solo complica el diagnóstico, sino que potencia la impulsividad y disminuye el umbral de inhibición, aumentando el riesgo de autolesión.
*   **Diferenciación:** A diferencia de los casos anteriores, el rol de "empresario" de Diego sugiere un entorno de alta presión que podría estar exacerbando el cuadro. Debemos descartar si el consumo de alcohol es una respuesta al estrés laboral o una patología dual preexistente.

### Paso 3: Evaluación preliminar
Mi impresión diagnóstica preliminar es un **Trastorno Depresivo Mayor (TDM), episodio actual moderado-grave, con comorbilidad por abuso de sustancias (alcohol).**
La desesperanza expresada y el insomnio severo son predictores críticos de riesgo suicida. El alcohol, al ser un depresor del sistema nervioso central, está exacerbando la sintomatología depresiva y creando un círculo vicioso que impide la recuperación natural del sueño y la estabilidad emocional.

### Paso 4: Plan de tratamiento y prioridades

**Prioridad 1: Evaluación de Seguridad (Inmediata)**
*   Realizar una evaluación de riesgo suicida estructurada (escala de Columbia o similar) en la primera sesión.
*   Determinar si es necesaria una internación breve (similar al Caso B) para desintoxicación y estabilización del sueño, dado que el insomnio de 2-3 horas es insostenible para un tratamiento ambulatorio efectivo.

**Prioridad 2: Estabilización Biológica**
*   Interconsulta con Psiquiatría para valorar el inicio de un esquema farmacológico (ISRS o duales) y, fundamentalmente, un hipnótico/sedante de corta duración para romper el ciclo de insomnio.
*   Abordaje del consumo de alcohol: Establecer un contrato de abstinencia o reducción de daños supervisada.

**Prioridad 3: Intervención Psicoterapéutica (TCC)**
*   **Fase inicial:** Psicoeducación sobre la relación entre alcohol, insomnio y depresión.
*   **Fase media:** Reestructuración cognitiva para abordar la desesperanza y entrenamiento en regulación emocional para manejar la irritabilidad.
*   **Fase de mantenimiento:** Identificación de estresores laborales y desarrollo de estrategias de afrontamiento adaptativas.

**Prioridad 4: Red de Apoyo**
*   Involucrar a la familia (con consentimiento del paciente) para asegurar el cumplimiento del tratamiento y monitoreo de riesgos en el hogar.

---
**Nota:** *Esta es una evaluación clínica preliminar. El paciente debe ser derivado de forma urgente a una evaluación psiquiátrica presencial para descartar riesgo inminente de daño a sí mismo o a terceros.*


[Tokens generados: 983]


## 6️⃣ Tu Turno: Crea tu Consulta

✏️ Inventa un paciente y aplica **al menos 2 técnicas** diferentes. Algunas ideas:
- 📚 Un estudiante con estres por examenes (Zero Shot + Role Prompting)
- 🏡 Un adulto mayor con aislamiento social (Few Shot + Chain of Thought)
- 📱 Un adolescente con adiccion al celular (Chain of Thought + Role Prompting)

In [ ]:
# Tu consulta personalizada
mi_prompt = """[ESCRIBE TU PROMPT AQUI]"""

# Descomenta cuando estes listo:
# consultar(mi_prompt, opciones={"temperature": 0.5, "num_predict": 2000})

### 📋 Resumen

| Técnica | Qué hace | Cuándo usarla |
|---|---|---|
| **Zero Shot** | Pide sin ejemplos | Tareas generales, primera consulta |
| **Few Shot** | Da ejemplos primero | Seguir un formato o patrón específico |
| **Chain of Thought** | Razona paso a paso | Análisis complejo, evaluaciones |
| **Role Prompting** | Asigna un rol | Perspectivas especializadas |

---

🏥 **Consultorio cerrado por hoy. Buen trabajo, doctores.** 👨‍⚕️👩‍⚕️